### Download dataset and convert to Pandas dataframe

In [2]:
# installed but not in requirements.txt, so check version
! pip show kagglehub | grep "Version" | head -n 1

Version: 1.0.2


In [33]:
import kagglehub
import pandas as pd

In [34]:
def download_dataset(year_start, year_end_exd,
                     origin_path="flnny123/mfddmulti-modal-flight-delay-dataset/versions/4",
                     ):
    dest_paths=[]
    for year in range(year_start, year_end_exd): 
        print(f"Downloading year {year} data...")
        origin_path_year = "Aeolus/Flight_Tab/flight_with_weather_" + str(year) + ".csv"
        dest_path_year = kagglehub.dataset_download(origin_path, path=origin_path_year)
        print(f"(CSV file available at {dest_path_year}).")
        dest_paths.append(dest_path_year)
    return dest_paths   # ora fuori dal ciclo: scarica tutti gli anni richiesti

def read_dataset_pandas(file_path_year, exploring=True):
    print(f"Converting to Pandas dataframe...")
    if exploring:
        df = pd.read_csv(file_path_year, nrows=10000)
    else:
        df = pd.read_csv(file_path_year)
    print("All done.")
    return df

In [35]:
dfs_paths = download_dataset(year_start=2022, year_end_exd=2025)  # 2022, 2023, 2024
print(f"{len(dfs_paths)} file scaricati")

(CSV file available at /home/sara/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Aeolus/Flight_Tab/flight_with_weather_2022.csv).


100%|██████████| 1.72G/1.72G [10:16<00:00, 3.00MB/s]


(CSV file available at /home/sara/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Aeolus/Flight_Tab/flight_with_weather_2023.csv).


100%|██████████| 1.63G/1.63G [10:22<00:00, 2.81MB/s]

(CSV file available at /home/sara/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Aeolus/Flight_Tab/flight_with_weather_2024.csv).
3 file scaricati


In [38]:
#df_2022_path=dfs_paths[0]
#df_2022= read_dataset_pandas(file_path_year=df_2022_path,
 #                            exploring=True) # if True, get limited number of rows to avoid memory issues):


### Explore

In [39]:
from IPython import display
#display.Image("/mnt/c/Users/Libero/uni/1B/nndl_final_project_theory/features_explained.jpeg")

In [40]:
# WHEELS_ON and WHEELS_OFF likely represent actual landing and takeoff times respectively
# (so they should be considered targets);
# also assuming all times are in the same timezone.

In [41]:
#type(df_2022)

In [42]:
#type(df_2022_path)

In [43]:
#df_2022.iloc[:,:15].tail()

In [44]:
#df_2022.iloc[:,15:].head()

In [45]:
#df_2022.sort_values(by="AIR_TIME").tail(1)["AIR_TIME"] # longest airtime

In [46]:
#df_2022.dtypes

### Clean up

In [47]:
# define column names as global variables
DATE_COLS=["FL_DATE"]
DATETIME_COLS=["CRS_DEP_TIME", "CRS_ARR_TIME", "DEP_TIME",
               "ARR_TIME", "WHEELS_OFF", "WHEELS_ON", ]
TIMEDELTA_MINS_COLS=["DEP_DELAY", "ARR_DELAY", "TAXI_OUT", "TAXI_IN", "CRS_ELAPSED_TIME",
                     "ACTUAL_ELAPSED_TIME", "AIR_TIME",	]
INT_COLS=["OP_CARRIER_FL_NUM", "FLIGHTS", "MONTH", "DAY_OF_MONTH",
          "DAY_OF_WEEK", "ORIGIN_INDEX", "DEST_INDEX"]
STR_COLS=["OP_CARRIER", "ORIGIN", "DEST"]
FLOAT_COLS=["O_TEMP", "O_PRCP", "O_WSPD", "D_TEMP", "D_PRCP", "D_WSPD", "O_LATITUDE",
             "O_LONGITUDE", "D_LATITUDE", "D_LONGITUDE"]
size_set_check=set(DATE_COLS+DATETIME_COLS+TIMEDELTA_MINS_COLS+INT_COLS+STR_COLS+FLOAT_COLS)
print(f"Total individual features (should be 34): {len(size_set_check)}")

Total individual features (should be 34): 34


In [48]:
def remove_outliers_percentile(df, columns, lower=0.01, upper=0.99): # good practice 
    n_rows_before=len(df)
    for col in columns:
        lower_bound = df[col].quantile(lower)
        upper_bound = df[col].quantile(upper)
        df= df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    n_rows_after=len(df)
    percentage_dropped=100 *(n_rows_before-n_rows_after)/n_rows_before 
    print(f"Dropped {percentage_dropped:.2f} % of rows because outliers.")
        
    return df

In [49]:
import numpy as np

In [50]:
def clean_dataframe(df,
                  int_type='int32', float_type='float32',
                  cache_bool=False,): # cache=False to save memory at the cost of speed
    # type conversions           
    for col in DATE_COLS+DATETIME_COLS:
        df[col] = pd.to_datetime(df[col], format="%Y-%m-%d %H:%M:%S", cache=cache_bool) 
    for col in DATE_COLS: 
            df[col] = df[col].dt.normalize() # keep date only, no time (cleaner)
    for col in TIMEDELTA_MINS_COLS:
        df[col] = pd.to_timedelta(df[col], unit='m')
    df[INT_COLS] = df[INT_COLS].astype(int_type)
    df[STR_COLS] = df[STR_COLS].astype('str')
    df[FLOAT_COLS] = df[FLOAT_COLS].astype(float_type)
    
    # drop all rows containing NaNs and keep track of them
    n_rows_before=len(df)
    df.dropna(subset=['D_TEMP', 'D_PRCP', 'D_WSPD'], inplace=True)
    n_rows_after=len(df)
    n_rows_dropped=n_rows_before-n_rows_after
    print(f"Dropped {n_rows_dropped} rows because of NaNs.")
    nan_bool = bool(np.any(df.isna().sum() > 0))   # era df_2022, ora usa il parametro df
    print(f"Any NaNs remaining in numerical data: {nan_bool}.")
    
    return df

In [51]:
#df_2022.dtypes

In [52]:
#df_2022.describe()

In [53]:
#df_2022.columns[np.where(df_2022.isna().sum()>0)]

In [54]:
#df_2022[pd.isna(df_2022['D_TEMP'])]

In [55]:
#df_2022[pd.isna(df_2022['D_PRCP'])]

In [56]:
#df_2022[pd.isna(df_2022['D_WSPD'])]

In [57]:
#df_2022.iloc[:,:15].head()

In [58]:
#df_2022.iloc[:,15:].head()

### Prepare (tabular) dataset for tabular / sequential experiments

In [29]:
from sklearn.preprocessing import LabelEncoder # to encode categorical features as integers

In [59]:
# IMPORTANT NOTE: this modifies dataframe in-place;
# if mistaken, go back to read_dataset_pandas(...)  
def prepare_df(df, mode="tabular", # tabular or sequential
               target_columns=["DEP_DELAY", "ARR_DELAY"],  # in minutes
               time_of_prediction="departure"): # departure or arrival (for tabular mode)
    
    if(mode=="tabular"):
        # NOTE: outlier removal moved downstream (train-only, after the temporal
        # split) to avoid leaking val/test bounds and to avoid double-removal
        df = df.dropna(subset=['FL_DATE'])
        df['FL_YEAR'] = df['FL_DATE'].dt.year # flight year...
        df.drop(columns=['FL_DATE'], inplace=True) #... instead of full date
        
        time_columns = ['CRS_DEP_TIME', 'DEP_TIME', 'WHEELS_OFF', 'WHEELS_ON',
                        'CRS_ARR_TIME', 'ARR_TIME'] 
        for col in time_columns:
            df[col+'_MIN'] = df[col].dt.hour * 60 + df[col].dt.minute
            # convert time columns in minutes since midnight...
        df.drop(columns=time_columns, inplace=True) #... and drop the originals

        categorical_columns = ['OP_CARRIER', 'OP_CARRIER_FL_NUM',
                            'FL_YEAR', 'MONTH', 'DAY_OF_MONTH', # unique identifier for any day;
                                                                # scheduled?? if not, minor leak
                            'ORIGIN', 'DEST']
        # NOTE: added weather features (were being dropped despite being cleaned
        # in clean_dataframe) and CRS_ELAPSED_TIME (also being dropped)
        if (time_of_prediction=="departure"):
            # info available before actual departure time
            continuous_columns = ['CRS_DEP_TIME_MIN','CRS_ARR_TIME_MIN', 'CRS_ELAPSED_TIME',
                                  'FLIGHTS',
                                  'O_TEMP', 'O_PRCP', 'O_WSPD', 'D_TEMP', 'D_PRCP', 'D_WSPD',
                                  'O_LATITUDE', 'O_LONGITUDE', 'D_LATITUDE', 'D_LONGITUDE']
        elif (time_of_prediction=="arrival"):
            # info available before actual arrival time (but after actual departure)
            continuous_columns = ['CRS_DEP_TIME_MIN','CRS_ARR_TIME_MIN', 'CRS_ELAPSED_TIME',
                                  'DEP_TIME_MIN', 'WHEELS_OFF_MIN', # <<<<<----------- new info
                                  'FLIGHTS',
                                  'O_TEMP', 'O_PRCP', 'O_WSPD', 'D_TEMP', 'D_PRCP', 'D_WSPD',
                                  'O_LATITUDE', 'O_LONGITUDE', 'D_LATITUDE', 'D_LONGITUDE']
        df = df[target_columns + categorical_columns + continuous_columns]

    elif(mode=="sequential"):
        # Feature engineering
        df['CRS_DEP_TIME_HOUR'] = df['CRS_DEP_TIME'].dt.hour.astype('int8')
        df['CRS_ARR_TIME_HOUR'] = df['CRS_ARR_TIME'].dt.hour.astype('int8')
        # Encode categorical features
        encoder = LabelEncoder()
        df['OP_CARRIER'] = encoder.fit_transform(df['OP_CARRIER'])
        df['OP_CARRIER_FL_NUM'] = encoder.fit_transform(df['OP_CARRIER_FL_NUM'])

        df['MONTH'] = df['FL_DATE'].dt.month
        df['DAY_OF_YEAR'] = df['FL_DATE'].dt.dayofyear
        dense_feat_cols = ['O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD', 'FLIGHTS']
        sparse_feat_cols = ['MONTH', 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR',
                             'CRS_DEP_TIME_HOUR', 'ORIGIN_INDEX', 'DEST_INDEX',
                             'OP_CARRIER', 'OP_CARRIER_FL_NUM']
        target_cols=['ARR_DELAY', 'DEP_DELAY']
        df=df[target_cols+["DAY_OF_YEAR"]+dense_feat_cols+sparse_feat_cols]
        # from here on will have to consider train/test split (without leaking), torch implementation, etc.
        # (group by to create multiple chains each identified by starting day of year, etc)
        # and also time of prediction




    return df


In [ ]:
#df_2022 = prepare_df(df_2022, mode="tabular", time_of_prediction="departure")

In [60]:
cleaned_dfs = []
for path in dfs_paths:
    df_year = read_dataset_pandas(file_path_year=path, exploring=False)  # tutte le righe, non 10.000
    df_year = clean_dataframe(df_year)
    df_year = prepare_df(df_year, mode="tabular", time_of_prediction="departure")
    cleaned_dfs.append(df_year)

df_all = pd.concat(cleaned_dfs, ignore_index=True)
print(f"Dataset finale: {df_all.shape}")

Converting to Pandas dataframe...
All done.
Dropped 667 rows because of NaNs.
Any NaNs remaining in numerical data: True.
Converting to Pandas dataframe...
All done.
Dropped 862 rows because of NaNs.
Any NaNs remaining in numerical data: True.
Converting to Pandas dataframe...
All done.
Dropped 750 rows because of NaNs.
Any NaNs remaining in numerical data: True.
Dataset finale: (19341439, 23)


In [61]:
df_all.to_parquet("aeolus_tabular_2022_2024.parquet", index=False)
print(f"Salvato: {df_all.shape}")

Salvato: (19341439, 23)


In [ ]:
#df_2022.iloc[:,:15].head()

In [ ]:
#df_2022.iloc[:,15:].head()

In [ ]:
#df_2022.dtypes